# 🔐 Privacy-Preserving ML — SRAE Training
### Google Colab Notebook (Run with GPU)

**Steps:**
1. Upload your dataset CSV
2. Run all cells in order
3. Download trained model
4. Put model in `saved_models/` folder

## CELL 1: Setup — Install Libraries

In [ ]:
# Install required libraries
!pip install tensorflow==2.5.0 scikit-learn numpy pandas matplotlib seaborn
print(' Libraries installed!')

##  CELL 2: Mount Google Drive

In [ ]:
# Mount Google Drive to save models
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/privacy_ml_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory: {SAVE_DIR}')

##  CELL 3: Upload Dataset

In [ ]:
from google.colab import files
import io, pandas as pd, numpy as np

print('Upload your CSV dataset file:')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'Dataset loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

##  CELL 4: Prepare Data

In [ ]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

# Features = all columns except last
# Labels   = last column
X = df.iloc[:, :-1].values.astype(np.float32)
y_raw = df.iloc[:, -1].values

# Encode string labels to numbers
le = LabelEncoder()
y = le.fit_transform(y_raw).astype(np.int32)
num_classes = len(le.classes_)

# Scale features to [0, 1]
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)

print(f'Input dim:   {X.shape[1]}')
print(f'Num classes: {num_classes} → {le.classes_}')
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

##  CELL 5: Build SRAE Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import tensorflow.keras.regularizers as reg

def residual_block(x, units):
    shortcut = x
    out = Dense(units, kernel_regularizer=reg.l1_l2(1e-5, 1e-5))(x)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dense(units, kernel_regularizer=reg.l1_l2(1e-5, 1e-5))(out)
    out = BatchNormalization()(out)
    if int(shortcut.shape[-1]) != units:
        shortcut = Dense(units)(shortcut)
        shortcut = BatchNormalization()(shortcut)
    out = Add()([out, shortcut])
    out = Activation('relu')(out)
    return out

def build_srae(input_dim, num_classes):
    inputs = Input(shape=(input_dim,))
    # Encoder
    psi_1 = residual_block(inputs, 256)
    psi_1 = Dense(256, activity_regularizer=reg.l1(1e-5), name='psi_1')(psi_1)
    psi_2 = residual_block(psi_1, 128)
    psi_2 = Dense(128, activity_regularizer=reg.l1(1e-5), name='psi_2')(psi_2)
    psi_3 = residual_block(psi_2, 96)
    psi_3 = Dense(96,  activity_regularizer=reg.l1(1e-5), name='psi_3')(psi_3)
    # Classifiers
    clf_1 = Dense(num_classes, activation='softmax', name='clf_1')(psi_1)
    clf_2 = Dense(num_classes, activation='softmax', name='clf_2')(psi_2)
    clf_3 = Dense(num_classes, activation='softmax', name='clf_3')(psi_3)
    # Concatenated encoding
    psi_concat = Concatenate(name='encoding')([psi_1, psi_2, psi_3])
    clf_final  = Dense(num_classes, activation='softmax', name='clf_final')(psi_concat)
    pca_proj   = Dense(2, name='pca_projection')(psi_concat)
    # Decoder
    dec = residual_block(psi_3, 128)
    dec = residual_block(dec, 256)
    recon = Dense(input_dim, activation='sigmoid', name='reconstruction')(dec)
    return Model(inputs, [recon, clf_1, clf_2, clf_3, clf_final, psi_concat, pca_proj])

model = build_srae(X_train.shape[1], num_classes)
print('Model built!')
print(f'Parameters: {model.count_params():,}')

##  CELL 6: Define Losses

In [ ]:
from sklearn.decomposition import PCA

class CenterLoss:
    def __init__(self, num_classes, feature_dim, alpha=0.5):
        self.alpha = alpha
        self.num_classes = num_classes
        self.centers = tf.Variable(tf.zeros([num_classes, feature_dim]),
                                   trainable=False)
    def __call__(self, features, labels):
        features = tf.cast(features, tf.float32)
        labels = tf.cast(labels, tf.int32)
        centers_batch = tf.gather(self.centers, labels)
        loss = tf.reduce_mean(tf.reduce_sum(tf.square(features - centers_batch), axis=1))
        for cid in range(self.num_classes):
            mask = tf.equal(labels, cid)
            cf = tf.boolean_mask(features, mask)
            if tf.shape(cf)[0] > 0:
                cm = tf.reduce_mean(cf, axis=0)
                diff = self.centers[cid] - cm
                nc = self.centers[cid] - self.alpha * diff
                self.centers[cid].assign(nc)
        return loss

def pca_loss(proj, x_pca):
    p = tf.nn.l2_normalize(tf.cast(proj, tf.float32), axis=1)
    q = tf.nn.l2_normalize(tf.cast(x_pca, tf.float32), axis=1)
    return tf.reduce_mean(1.0 - tf.reduce_sum(p * q, axis=1))

# Pre-compute PCA
pca = PCA(n_components=2)
x_pca_train = pca.fit_transform(X_train).astype(np.float32)

center_loss_fn = CenterLoss(num_classes, 480)
optimizer = tf.keras.optimizers.Adam(0.001)
print('Losses ready!')

##  CELL 7: Train the Model (GPU Accelerated!)

In [ ]:
EPOCHS = 100
BATCH  = 32
best_val = float('inf')
patience = 10
wait = 0
history = []

print(f'Training for {EPOCHS} epochs on GPU...')
print('='*60)

for epoch in range(EPOCHS):
    idx = np.random.permutation(len(X_train))
    nb  = max(1, len(X_train) // BATCH)
    el  = 0
    for b in range(nb):
        s, e = b*BATCH, (b+1)*BATCH
        xb = X_train[idx[s:e]]
        yb = y_train[idx[s:e]].astype(np.int32)
        pb = x_pca_train[idx[s:e]]
        with tf.GradientTape() as tape:
            out = model(xb, training=True)
            L1 = tf.reduce_mean(tf.square(xb - out[0]))
            L2 = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(yb, out[4]))
            L3 = center_loss_fn(out[5], yb)
            L4 = pca_loss(out[6], pb)
            total = L1 + L2 + 0.5*L3 + 0.5*L4
        grads = tape.gradient(total, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        el += total.numpy()
    el /= nb
    val_out = model(X_val, training=False)
    vl = tf.reduce_mean(tf.square(X_val - val_out[0])).numpy()
    history.append({'epoch': epoch+1, 'loss': el, 'val_loss': vl})
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {el:.4f} | ValLoss: {vl:.4f}')
    if vl < best_val:
        best_val = vl
        model.save(f'{SAVE_DIR}/srae_best.h5')
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f'Early stop at epoch {epoch+1}')
            break

print('\n Training complete!')
print(f'Best model saved to: {SAVE_DIR}/srae_best.h5')

##  CELL 8: Generate Encodings

In [ ]:
def get_encoding(model, X, batch=256):
    encs = []
    for i in range(0, len(X), batch):
        out = model(X[i:i+batch], training=False)
        encs.append(out[5].numpy())
    return np.vstack(encs)

psi_train = get_encoding(model, X_train)
psi_test  = get_encoding(model, X_test)

print(f'Encoding shape: {psi_train.shape}')  # Should be (N, 480)

# Save everything
np.save(f'{SAVE_DIR}/psi_train.npy', psi_train)
np.save(f'{SAVE_DIR}/psi_test.npy',  psi_test)
np.save(f'{SAVE_DIR}/y_train.npy',   y_train)
np.save(f'{SAVE_DIR}/y_test.npy',    y_test)
print(' Encodings saved!')

##  CELL 9: Evaluate Performance

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

X_base_train = psi_train[:, 384:]  # Baseline: latent only (last 96 dims)
X_base_test  = psi_test[:, 384:]

classifiers = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(C=1, kernel='rbf'),
    'RF':  RandomForestClassifier(n_estimators=200, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(200,100), max_iter=500, random_state=42)
}

print(f'{'Model':<8} {'Original':>12} {'Baseline':>12} {'Encoding':>12} {'Improve':>10}')
print('-'*58)
for name, clf in classifiers.items():
    # Original
    clf.fit(X_train, y_train)
    f_org = f1_score(y_test, clf.predict(X_test), average='macro')
    # Baseline
    clf.fit(X_base_train, y_train)
    f_base = f1_score(y_test, clf.predict(X_base_test), average='macro')
    # Encoding
    clf.fit(psi_train, y_train)
    f_enc = f1_score(y_test, clf.predict(psi_test), average='macro')
    print(f'{name:<8} {f_org*100:>11.2f}% {f_base*100:>11.2f}% {f_enc*100:>11.2f}% {(f_enc-f_base)*100:>+9.2f}%')

##  CELL 10: Plot Results

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(hist_df['loss'],     label='Train Loss', color='blue')
plt.plot(hist_df['val_loss'], label='Val Loss',   color='red')
plt.title('Training History')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True, alpha=0.3)

from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
enc_2d = tsne.fit_transform(psi_test[:min(500,len(psi_test))])
plt.subplot(1,2,2)
plt.scatter(enc_2d[:,0], enc_2d[:,1],
            c=y_test[:min(500,len(y_test))],
            cmap='tab10', alpha=0.6, s=15)
plt.title('t-SNE of Encoding Ψ (test set)')
plt.colorbar(label='Class')
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/results.png', dpi=100)
plt.show()
print(' Plots saved!')

##  CELL 11: Download Model

In [ ]:
# Download the trained model to your computer
# Then put it in your project's saved_models/ folder
files.download(f'{SAVE_DIR}/srae_best.h5')
files.download(f'{SAVE_DIR}/psi_train.npy')
files.download(f'{SAVE_DIR}/psi_test.npy')
files.download(f'{SAVE_DIR}/results.png')
print(' All files downloaded!')